# MongoDB Intermediate: How to Analyze Data at Scale

## Academic Performance Analytics

**Difficulty: Intermediate | ~40 min | Requires Labs 1 and 2**

*Lab 3 of 7 in the MongoDB Mastery series.*

In this lab, you will use MongoDB's aggregation pipeline to compute analytics and learn how indexing speeds up queries.

You will learn how to:
1. Connect to a MongoDB Atlas cluster
2. Compute average grades per course using `$group` and `$avg`
3. Create grade distributions using `$bucket`
4. Create an index and verify it with `.explain()`
5. Generate a formatted analytics summary report

In [11]:
!pip install -qU "pymongo[srv,tls]==4.10.1" python-dotenv==1.0.1 certifi


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


This installs the MongoDB Python driver (`pymongo`) with `srv` and `tls` extras, `python-dotenv` for loading credentials, and `certifi` for up-to-date CA certificates.

### Step 1 — Connect to MongoDB

In [12]:
import os
import certifi
from dotenv import load_dotenv
import pymongo

# Load the Atlas connection string from the .env file one directory up
load_dotenv("../.env")
uri = os.environ["MONGODB_URI"]

# Connect to the real MongoDB Atlas cluster
client = pymongo.MongoClient(uri, tlsCAFile=certifi.where())

# Access the database and collection
db = client["school_db"]
students = db["students"]

print("Connected to MongoDB Atlas")

Connected to MongoDB Atlas


Same connection pattern as Labs 1 and 2 — `load_dotenv` reads from the shared `.env` file, and `certifi.where()` provides trusted CA certificates for SSL.

### Step 2 — Insert the Starting Dataset

In [13]:
students.drop()

student_records = [
    {"name": "Alice Johnson",  "student_id": "STU001", "course": "Computer Science", "grade": 95, "enrollment_date": "2024-09-01", "status": "active"},
    {"name": "Bob Smith",      "student_id": "STU002", "course": "Mathematics",      "grade": 78, "enrollment_date": "2024-09-01", "status": "active"},
    {"name": "Charlie Brown",  "student_id": "STU003", "course": "Physics",          "grade": 55, "enrollment_date": "2024-09-02", "status": "active"},
    {"name": "Diana Prince",   "student_id": "STU004", "course": "English",          "grade": 48, "enrollment_date": "2024-09-01", "status": "active"},
    {"name": "Eve Torres",     "student_id": "STU005", "course": "Biology",          "grade": 88, "enrollment_date": "2024-09-03", "status": "active"},
    {"name": "Frank Castle",   "student_id": "STU006", "course": "Mathematics",      "grade": 52, "enrollment_date": "2024-09-01", "status": "inactive"},
    {"name": "Grace Hopper",   "student_id": "STU007", "course": "Computer Science", "grade": 91, "enrollment_date": "2024-09-02", "status": "active"},
    {"name": "Hank Pym",       "student_id": "STU008", "course": "Physics",          "grade": 73, "enrollment_date": "2024-09-01", "status": "active"},
    {"name": "Ivan Petrov",    "student_id": "STU009", "course": "Computer Science", "grade": 84, "enrollment_date": "2024-09-03", "status": "active"},
    {"name": "Julia Child",    "student_id": "STU010", "course": "English",          "grade": 90, "enrollment_date": "2024-09-02", "status": "graduated"},
    {"name": "Karl Marx",      "student_id": "STU011", "course": "Mathematics",      "grade": 67, "enrollment_date": "2024-09-01", "status": "active"},
    {"name": "Laura Palmer",   "student_id": "STU012", "course": "Biology",          "grade": 45, "enrollment_date": "2024-09-02", "status": "active"},
    {"name": "Marco Polo",     "student_id": "STU013", "course": "Physics",          "grade": 82, "enrollment_date": "2024-09-03", "status": "active"},
    {"name": "Nina Simone",    "student_id": "STU014", "course": "English",          "grade": 76, "enrollment_date": "2024-09-01", "status": "active"},
    {"name": "Oscar Wilde",    "student_id": "STU015", "course": "Mathematics",      "grade": 93, "enrollment_date": "2024-09-02", "status": "active"},
    {"name": "Pia Zadora",     "student_id": "STU016", "course": "Biology",          "grade": 71, "enrollment_date": "2024-09-01", "status": "active"},
    {"name": "Quincy Adams",   "student_id": "STU017", "course": "Computer Science", "grade": 87, "enrollment_date": "2024-09-03", "status": "active"},
    {"name": "Rosa Parks",     "student_id": "STU018", "course": "English",          "grade": 58, "enrollment_date": "2024-09-02", "status": "inactive"},
    {"name": "Sam Wilson",     "student_id": "STU019", "course": "Physics",          "grade": 98, "enrollment_date": "2024-09-01", "status": "active"},
    {"name": "Tina Turner",    "student_id": "STU020", "course": "Mathematics",      "grade": 79, "enrollment_date": "2024-09-03", "status": "active"},
    {"name": "Uma Thurman",    "student_id": "STU021", "course": "Computer Science", "grade": 62, "enrollment_date": "2024-09-02", "status": "active"},
    {"name": "Vera Wang",      "student_id": "STU022", "course": "Biology",          "grade": 85, "enrollment_date": "2024-09-01", "status": "graduated"},
    {"name": "Walt Disney",    "student_id": "STU023", "course": "Physics",          "grade": 56, "enrollment_date": "2024-09-02", "status": "active"},
    {"name": "Xena Warrior",   "student_id": "STU024", "course": "English",          "grade": 94, "enrollment_date": "2024-09-03", "status": "active"},
    {"name": "Yusuf Islam",    "student_id": "STU025", "course": "Computer Science", "grade": 70, "enrollment_date": "2024-09-01", "status": "active"},
]

result = students.insert_many(student_records)
print(f"Inserted {len(result.inserted_ids)} student records.")

Inserted 25 student records.


`students.drop()` ensures re-running the notebook starts clean without duplicating records.

### Step 3 — Aggregation: Average Grade per Course

In [14]:
# Aggregation pipeline: group students by course, compute average grade, sort descending
pipeline = [
    {"$group": {"_id": "$course", "avg_grade": {"$avg": "$grade"}}},
    {"$sort": {"avg_grade": -1}}
]

course_avgs = list(students.aggregate(pipeline))

print("--- Average Grade per Course ---")
for doc in course_avgs:
    print(f"{doc['_id']:<20} | Avg: {doc['avg_grade']:.1f}")

--- Average Grade per Course ---
Computer Science     | Avg: 81.5
Mathematics          | Avg: 73.8
English              | Avg: 73.2
Physics              | Avg: 72.8
Biology              | Avg: 72.2


`$group` collects all students sharing the same `course` value. `$avg` computes the mean grade within each group. `$sort` orders the results from highest to lowest average.

### Step 4 — Aggregation: Grade Distribution with $bucket

In [15]:
# $bucket distributes grades into ranges: F (0-59), D (60-69), C (70-79), B (80-89), A (90-100)
pipeline = [
    {"$bucket": {
        "groupBy": "$grade",
        "boundaries": [0, 60, 70, 80, 90, 101],
        "default": "Other",
        "output": {"count": {"$sum": 1}}
    }}
]

distribution = list(students.aggregate(pipeline))

print("--- Grade Distribution ---")
labels = {0: "0-59   (F)", 60: "60-69  (D)", 70: "70-79  (C)",
          80: "80-89  (B)", 90: "90-100 (A)"}
for doc in distribution:
    label = labels.get(doc["_id"], doc["_id"])
    print(f"  {label}: {doc['count']} students")

--- Grade Distribution ---
  0-59   (F): 6 students
  60-69  (D): 2 students
  70-79  (C): 6 students
  80-89  (B): 5 students
  90-100 (A): 6 students


`$bucket` distributes documents into predefined ranges based on the `grade` field. Each `boundaries` value is the inclusive lower edge of a bucket. The `output` key defines what to compute for each bucket — here, a simple count.

### Step 5 — Query Performance Without an Index

In [ ]:
query = {"course": "Computer Science"}
projection = {"_id": 0, "course": 1}

# Before creating any index, run the query and inspect the execution plan
# Project only the indexed field so the query is covered (no FETCH needed)
plan_before = students.find(query, projection).explain()

winning_before = plan_before.get("queryPlanner", {}).get("winningPlan", {})
stage_before = winning_before.get("inputStage", {}).get("stage", winning_before.get("stage", "unknown"))

print("--- Before indexing ---")
print(f"Stage: {stage_before}")

With no index on `course`, MongoDB must perform a **collection scan** (COLLSCAN) — it reads every document in the collection to find matches. This is fine for 25 documents, but would be slow on a collection with millions.

### Step 6 — Create an Index and Verify the Difference

In [ ]:
students.create_index("course")
print("Index created on 'course' field")

# Now run the same covered query again and inspect the execution plan
plan_after = students.find(query, projection).explain()

winning_after = plan_after.get("queryPlanner", {}).get("winningPlan", {})
stage_after = winning_after.get("inputStage", {}).get("stage", winning_after.get("stage", "unknown"))

print(f"\n--- After indexing ---")
print(f"Stage: {stage_after}")

print(f"\n--- Comparison ---")
print(f"Before index: {stage_before}")
print(f"After index:  {stage_after}")

Creating an index on `course` lets MongoDB locate matching documents via an **index scan** (IXSCAN) instead of scanning every document. By projecting only the indexed field (`{"_id": 0, "course": 1}`), the query becomes **covered** — MongoDB can answer it entirely from the index without fetching the full document. We read the scan stage from `inputStage` (the inner stage of the plan) rather than the top-level `stage`, because MongoDB may wrap the scan in a projection stage. The result: the `inputStage` stage changes from `COLLSCAN` to `IXSCAN` after the index is created.

### Step 7 — Analytics Summary Report

In [18]:
# Overall stats
total = students.count_documents({})
pipeline_all = [{"$group": {"_id": None, "avg": {"$avg": "$grade"}}}]
overall_avg = list(students.aggregate(pipeline_all))[0]["avg"]

# Top and bottom courses
top_course = course_avgs[0]
bottom_course = course_avgs[-1]

# Grade distribution (same pipeline as Step 4)
dist_pipeline = [
    {"$bucket": {
        "groupBy": "$grade",
        "boundaries": [0, 60, 70, 80, 90, 101],
        "output": {"count": {"$sum": 1}}
    }}
]
dist = {doc["_id"]: doc["count"] for doc in students.aggregate(dist_pipeline)}

print("       ACADEMIC PERFORMANCE ANALYTICS")
print(f"\nTotal students: {total}")
print(f"Overall average grade: {overall_avg:.1f}")

print(f"\nTop course:    {top_course['_id']} (avg {top_course['avg_grade']:.1f})")
print(f"Bottom course: {bottom_course['_id']} (avg {bottom_course['avg_grade']:.1f})")

print("\n--- Grade Distribution ---")
for bound, label in [(90, "A"), (80, "B"), (70, "C"), (60, "D"), (0, "F")]:
    print(f"  {label}: {dist.get(bound, 0)} students")

       ACADEMIC PERFORMANCE ANALYTICS

Total students: 25
Overall average grade: 75.1

Top course:    Computer Science (avg 81.5)
Bottom course: Biology (avg 72.2)

--- Grade Distribution ---
  A: 6 students
  B: 5 students
  C: 6 students
  D: 2 students
  F: 6 students


Collects the key metrics from each aggregation step into one formatted summary — the kind of report an academic office would review at the end of a semester.